# Course 8: Research Engineering Interviews
## Session 6: Debugging rounds

**Goal:** find seven planted bugs in a short training script, using cheap checks in a fixed order.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** the tests and reference solutions here were run once on a CPU (November 2026). Cells marked TODO are stubs for you; try them under a timer before opening the reference solutions at the bottom.

### The script
It trains a small classifier on three well-separated clusters, so a correct script should score well above 90%. It runs without errors and scores near chance. It contains **seven** bugs. Find them in 30 minutes, thinking aloud, and write down the check that exposed each.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

def make_data(seed=0):
    g = torch.Generator().manual_seed(seed)
    centers = torch.tensor([[0.0, 0.0], [4.0, 0.0], [2.0, 3.5]]) + 100.0     # features far from zero
    y = torch.randint(0, 3, (900,), generator=g)
    X = centers[y] + torch.randn(900, 2, generator=g)
    return X[:600], y[:600], X[600:], y[600:]

def buggy_run():
    torch.manual_seed(0)
    Xtr, ytr, Xte, yte = make_data()
    mean, std = Xte.mean(0), Xte.std(0)
    Xtr, Xte = (Xtr - mean) / std, (Xte - mean) / std
    Xtr = Xtr[torch.randperm(len(Xtr))]
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Dropout(0.5), nn.Linear(32, 3))
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Dropout(0.5), nn.Linear(32, 3))
    for epoch in range(20):
        for i in range(0, len(Xtr), 64):
            xb, yb = Xtr[i:i + 64], ytr[i:i + 64]
            loss = F.cross_entropy(torch.softmax(model(xb), dim=-1), yb)
            loss.backward()
            opt.step()
    accs = []
    for i in range(0, len(Xte), 64):
        pred = model(Xte[i:i + 64]).argmax(-1)
        accs.append((pred == yte[i:i + 64]).float().mean().item())
    return sum(accs) / len(accs)

print("buggy test accuracy:", round(buggy_run(), 3))

### Cheap checks to run first
1. Read the six parts: data, model, loss, optimizer, loop, evaluation.
2. Is the loss at initialization near ln 3 = 1.10?
3. Can the model overfit one batch of 16?
4. Do the weights change after one optimizer step?
5. Are inputs and labels still aligned after any shuffle?
6. Whose statistics normalize the data?
7. Is the model in evaluation mode when it is evaluated, and how is the metric averaged?

In [ ]:
# Example check: do the weights change after a step?  Adapt this to the script above.
model = nn.Linear(2, 3); other = nn.Linear(2, 3)
opt = torch.optim.SGD(other.parameters(), lr=0.1)        # optimizer holds the wrong model
before = model.weight.clone()
F.cross_entropy(model(torch.randn(8, 2)), torch.randint(0, 3, (8,))).backward()
opt.step()
print("weights changed:", not torch.equal(before, model.weight))

In [ ]:
def my_fixed_run():
    raise NotImplementedError      # TODO: copy buggy_run here and fix it

# print(my_fixed_run())

### Lab: plant your own
Take a working MNIST or small-GPT script, plant eight bugs, keep an answer key, and swap with a partner.

---
## Reference solutions
Try every problem under the timer first. Then run this section: it defines working versions and runs the same tests.

In [ ]:
def fixed_run():
    torch.manual_seed(0)
    Xtr, ytr, Xte, yte = make_data()
    mean, std = Xtr.mean(0), Xtr.std(0)                    # 1. statistics from the training set
    Xtr, Xte = (Xtr - mean) / std, (Xte - mean) / std
    perm = torch.randperm(len(Xtr))
    Xtr, ytr = Xtr[perm], ytr[perm]                        # 2. shuffle inputs and labels together
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Dropout(0.5), nn.Linear(32, 3))
    opt = torch.optim.SGD(model.parameters(), lr=0.1)      # 3. optimizer built on the model that is trained
    for epoch in range(20):
        for i in range(0, len(Xtr), 64):
            xb, yb = Xtr[i:i + 64], ytr[i:i + 64]
            loss = F.cross_entropy(model(xb), yb)          # 4. logits, no softmax, into cross_entropy
            opt.zero_grad()                                # 5. clear old gradients
            loss.backward()
            opt.step()
    model.eval()                                           # 6. dropout off for evaluation
    with torch.no_grad():
        pred = model(Xte).argmax(-1)
    return (pred == yte).float().mean().item()             # 7. accuracy over all examples, not per batch

print("fixed test accuracy:", round(fixed_run(), 3))

**The seven bugs:** normalization statistics taken from the test set; inputs shuffled without their labels; the optimizer built on a model that is then replaced; a softmax applied before `cross_entropy`; no `zero_grad`; no `model.eval()` so dropout stays on at test time; accuracy averaged per batch although the last batch is smaller.